In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar
from datetime import datetime, timezone, timedelta
import logging
import pickle
import math
import asyncio


logging.basicConfig(
    level=logging.INFO,  # Set the logging level
    format='%(asctime)s - %(levelname)s - %(message)s',  # Format for the log messages
    handlers=[
        logging.StreamHandler()  # Log to the console
    ]
)

%reload_ext autoreload
%autoreload 2
from data.raw.retrievers.alpaca_markets_retriever import AlpacaMarketsRetriever
from config.constants import *
from data.raw.retrievers.alpaca_portfolio_selection import (
    calculate_portfolio_cap_share,
    get_daily_stats,
    inspect_portfolio_history_depths,
    select_portfolio,
)
from data.raw.retrievers.stooq_utils import prepare_data_stooq
from data.processed.dataset_creation import DatasetCreator
from data.processed.indicators import *
from data.processed.targets import Balanced3ClassClassification
from data.processed.normalization import ZScoreOverWindowNormalizer, ZScoreNormalizer, MinMaxNormalizer
from data.processed.missing_values_handling import DummyMissingValuesHandler
from data.processed.dataset_pytorch import DatasetPytorch
from modeling.trainer import Trainer

from config.train_config import load_train_config
from config.constants import *

config = load_train_config()


In [2]:
from alpaca.data.timeframe import TimeFrame

retriever = AlpacaMarketsRetriever()
all_symbols = retriever.get_all_symbols()
len(all_symbols)

4880

In [3]:
portfolio, skipped_assets = await select_portfolio(
    all_symbols,
    start_date=config.data_config.end - timedelta(days=120),
    end_date=config.data_config.end,
    min_history_depth=datetime(2024, 11, 1, tzinfo=Constants.Data.EASTERN_TZ),
    portfolio_size=100,
    criteria='E_1m',
)
portfolio, skipped_assets

2026-08-10 23:26:06,402 - INFO - Starting performance sweep for 83 days...
2026-08-10 23:26:06,403 - INFO - Processing day 2026-04-03 00:00:00-04:00
2026-08-10 23:26:28,786 - WARNING - Error on 2026-04-03 (Attempt 1/3): division by zero
2026-08-10 23:26:28,787 - INFO - Retrying in 5 seconds...
2026-08-10 23:26:58,423 - WARNING - Error on 2026-04-03 (Attempt 2/3): division by zero
2026-08-10 23:26:58,423 - INFO - Retrying in 10 seconds...
2026-08-10 23:28:19,641 - WARNING - Error on 2026-04-03 (Attempt 3/3): division by zero
2026-08-10 23:28:19,642 - ERROR - Failed to retrieve data for 2026-04-03 after 3 attempts. Skipping.
2026-08-10 23:28:19,646 - INFO - Processing day 2026-04-06 00:00:00-04:00
2026-08-10 23:29:28,959 - INFO - Retrieving Alpaca bars from the Alpaca API for 2026-04-06 09:30:00-04:00 to 2026-04-06 16:00:00-04:00.
2026-08-10 23:29:31,729 - INFO - Retrieving Alpaca bars from the Alpaca API for 2026-04-06 09:30:00-04:00 to 2026-04-06 16:00:00-04:00.
2026-08-10 23:29:34,343

([('QQQ', 4.088766735171041),
  ('SPY', 4.082437367589693),
  ('NVDA', 3.796267805373124),
  ('IWM', 3.067198941326103),
  ('INTC', 2.669121263337239),
  ('TQQQ', 2.5815097649283203),
  ('AAPL', 2.55033068648906),
  ('TSLA', 2.1140093035921548),
  ('XLK', 2.038483554385465),
  ('GOOGL', 2.013726470589105),
  ('NFLX', 1.8665160976630428),
  ('AMZN', 1.8646938873461543),
  ('IVV', 1.8115153050334905),
  ('VOO', 1.7782503471498676),
  ('MSFT', 1.7777059942771647),
  ('MU', 1.755423825334379),
  ('IREN', 1.7211483182967053),
  ('PLTR', 1.6918667017419091),
  ('RSP', 1.6848008281987035),
  ('GOOG', 1.6655763510569908),
  ('SOXX', 1.6357918866865502),
  ('XLV', 1.6127429467250516),
  ('SMH', 1.5954181025004952),
  ('HOOD', 1.5633651036353204),
  ('DIA', 1.5411929335188328),
  ('XLY', 1.4960326963504866),
  ('GDX', 1.4905072358131597),
  ('WULF', 1.4531990762168272),
  ('VTI', 1.4272011927000903),
  ('IGV', 1.419545008675146),
  ('BKNG', 1.406843943736229),
  ('XLI', 1.4036562783444393),
  ('

In [5]:
porfolio_sorted = sorted([asset for asset, score in portfolio])
porfolio_sorted

['AAPL',
 'ACWI',
 'AMD',
 'AMZN',
 'APLD',
 'ARKK',
 'AVGO',
 'B',
 'BAC',
 'BKNG',
 'BMY',
 'BSX',
 'CIFR',
 'CSCO',
 'DIA',
 'DVN',
 'EEM',
 'EFA',
 'EMXC',
 'EWT',
 'EWY',
 'FBTC',
 'FCX',
 'GDX',
 'GLD',
 'GOOG',
 'GOOGL',
 'HOOD',
 'IBIT',
 'IEMG',
 'IGV',
 'IJR',
 'INTC',
 'IONQ',
 'IREN',
 'IVV',
 'IVW',
 'IWF',
 'IWM',
 'IYR',
 'KO',
 'KRE',
 'MARA',
 'META',
 'MRVL',
 'MSFT',
 'MSTR',
 'MU',
 'NEE',
 'NFLX',
 'NKE',
 'NOW',
 'NVDA',
 'ORCL',
 'OXY',
 'PLTR',
 'PYPL',
 'QBTS',
 'QLD',
 'QQQ',
 'QQQM',
 'RGTI',
 'RIOT',
 'RKLB',
 'RSP',
 'SHEL',
 'SLB',
 'SLV',
 'SMCI',
 'SMH',
 'SOXX',
 'SPXL',
 'SPY',
 'SPYG',
 'STM',
 'TFC',
 'TQQQ',
 'TSLA',
 'TSM',
 'UBER',
 'UPRO',
 'USB',
 'VNQ',
 'VOO',
 'VT',
 'VTI',
 'VTWO',
 'VUG',
 'WFC',
 'WMT',
 'WULF',
 'XBI',
 'XLC',
 'XLE',
 'XLI',
 'XLK',
 'XLP',
 'XLV',
 'XLY',
 'XOM']

In [4]:
inspect_portfolio_history_depths(portfolio, retriever)

SPY 2016-01-01 00:01:00+00:00
QQQ 2016-01-01 00:00:00+00:00
NVDA 2016-01-04 11:37:00+00:00
TQQQ 2016-01-01 00:06:00+00:00
DRAM 2016-01-04 16:38:00+00:00
AAPL 2016-01-01 00:00:00+00:00
IWM 2016-01-01 00:11:00+00:00
INTC 2016-01-01 00:48:00+00:00
GOOGL 2016-01-04 09:00:00+00:00
VOO 2016-01-04 13:00:00+00:00
IREN 2021-11-17 16:44:00+00:00
XLK 2016-01-04 11:57:00+00:00
NFLX 2016-01-01 00:03:00+00:00
IVV 2016-01-04 12:02:00+00:00
AMZN 2016-01-01 00:56:00+00:00
MU 2016-01-01 00:12:00+00:00
EWY 2016-01-04 14:28:00+00:00
GDX 2016-01-01 00:15:00+00:00
TSLA 2016-01-01 00:30:00+00:00
SMH 2016-01-04 14:30:00+00:00
XLV 2016-01-04 13:00:00+00:00
RSP 2016-01-04 14:30:00+00:00
MSFT 2016-01-01 00:02:00+00:00
PLTR 2020-09-30 17:38:00+00:00
SOXX 2016-01-04 13:46:00+00:00
WMT 2016-01-04 09:55:00+00:00
KLAC 2016-01-04 14:30:00+00:00
XLY 2016-01-04 13:44:00+00:00
QQQM 2020-10-13 13:38:00+00:00
DIA 2016-01-01 00:08:00+00:00
WULF 2016-01-05 15:37:00+00:00
VTI 2016-01-04 13:00:00+00:00
SNDK 2025-02-13 14:42:00

['SPY',
 'QQQ',
 'NVDA',
 'TQQQ',
 'DRAM',
 'AAPL',
 'IWM',
 'INTC',
 'GOOGL',
 'VOO',
 'IREN',
 'XLK',
 'NFLX',
 'IVV',
 'AMZN',
 'MU',
 'EWY',
 'GDX',
 'TSLA',
 'SMH',
 'XLV',
 'RSP',
 'MSFT',
 'PLTR',
 'SOXX',
 'WMT',
 'KLAC',
 'XLY',
 'QQQM',
 'DIA',
 'WULF',
 'VTI',
 'SNDK',
 'GLD',
 'BKNG',
 'EWT',
 'IWF',
 'SMCI',
 'KO',
 'ASX',
 'KRE',
 'HOOD',
 'CSCO',
 'GOOG',
 'NOW',
 'IONQ',
 'VGT',
 'QLD',
 'CRWV',
 'CIFR',
 'XLI',
 'SLV',
 'XLC',
 'STM',
 'IYR',
 'ARKK',
 'IGV',
 'VUG',
 'XLE',
 'BMY',
 'BAC',
 'EEM',
 'APLD',
 'IJR',
 'IVW',
 'TSM',
 'IEMG',
 'VZ',
 'AVGO',
 'XLP',
 'EFA',
 'IEFA',
 'VNQ',
 'VTWO',
 'XOM',
 'BKR',
 'BSX',
 'RKLB',
 'RIOT',
 'FIG',
 'QBTS',
 'EWJ',
 'SPYG',
 'CSX',
 'BMNR',
 'NKE',
 'MARA',
 'NEE',
 'MRVL',
 'AMD',
 'PYPL',
 'FCX',
 'MSTR',
 'CTSH',
 'SOXQ',
 'SLB',
 'HPE',
 'ORCL',
 'NBIS',
 'DVN']

In [5]:
await calculate_portfolio_cap_share(portfolio, config.data_config.end)

ZeroDivisionError: division by zero